# Week 4b — A physics-informed reference correction for transfer between ions
**The problem (Week 4):** for an ion never seen in training, the errors for Zn, Ca and Li were almost entirely a **constant offset** (MAE ≈ |bias|). The model ranks the materials correctly but puts the whole voltage scale of the new ion in the wrong place.

**The physics:** every DFT voltage is measured against the bulk metal of *its own* ion, roughly

$$V_{\text{ion}} \approx E_{\text{host}} - E^\circ_{\text{ion}}$$

where $E^\circ$ is the metal's standard electrode potential (V vs SHE). Zn metal ($E^\circ = -0.76$ V) is far more stable than Li metal ($-3.04$ V), so the same host gives a much lower voltage with Zn.

**The idea:** train on the host-side potential $E_{\text{host}} = V + E^\circ_{\text{ion}}$, which should be roughly the same for every ion, then convert back with the target ion's own $E^\circ$. This shift uses only known physical constants, so it works even for an ion with **no training data at all**.

**This notebook compares, for each ion left out of training:**
1. **Direct** — predict the voltage directly (Week 4)
2. **E°-shifted** — train on $V + E^\circ$, convert back
3. **Oracle offset** — the best possible constant correction, using the true bias (not available in practice; it shows the ceiling for any offset correction)

Plus a small few-shot test for Mg and Zn. Uses files from Weeks 2–3; no API key needed.

In [ ]:
!pip -q install xgboost
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/electrode_voltage_ml"

In [ ]:
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

ION_ORDER = ["Li", "Na", "K", "Mg", "Ca", "Zn"]
PALETTE = dict(zip(ION_ORDER, sns.color_palette("tab10", len(ION_ORDER))))
# Standard electrode potentials, V vs SHE (M^z+ + z e- -> M)
E0 = {"Li": -3.04, "Na": -2.71, "K": -2.93, "Mg": -2.37, "Ca": -2.87, "Zn": -0.76}

feat = pd.read_csv(f"{DATA_DIR}/features_week2b.csv")
FP = pd.read_csv(f"{DATA_DIR}/chgnet_fingerprints.csv")
params = json.load(open(f"{DATA_DIR}/xgb_params.json"))["best_params"]
y = feat["average_voltage"].values
ions = feat["working_ion"].values
groups = feat["framework"].values
shift = pd.Series(ions).map(E0).values                     # E° of each electrode's ion
X = pd.concat([feat.drop(columns=["average_voltage", "working_ion", "framework"]), FP], axis=1)   # Combined features

try:
    import torch; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
def new_model():
    return XGBRegressor(**params, min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1,
                        tree_method="hist", device=DEVICE)
print(f"{len(y)} electrodes, {X.shape[1]} features, XGBoost on {DEVICE}")

## 1. Does the shift make voltages more similar across ions?
If the physics is right, the spread of the *ion-averaged* values should shrink after adding $E^\circ$: hosts should look alike regardless of the ion.

In [ ]:
tab = pd.DataFrame({"mean V (vs own metal)": pd.Series(y).groupby(ions).mean(),
                    "mean V + E° (vs SHE)": pd.Series(y + shift).groupby(ions).mean(),
                    "E°": pd.Series(E0)}).reindex(ION_ORDER).round(2)
print(tab)
print(f"\nSpread of ion means: {tab.iloc[:, 0].std():.2f} V before shift, {tab.iloc[:, 1].std():.2f} V after shift")

## 2. Zero-shot transfer: direct vs. E°-shifted
For each target ion we train on all other ions (known hosts, and the strict new-hosts setting) and predict every electrode of the target ion.

In [ ]:
def evaluate(t, p):
    bias = float(np.mean(p - t))
    return {"MAE": mean_absolute_error(t, p), "bias": bias, "R2": r2_score(t, p),
            "MAE_oracle_offset": mean_absolute_error(t, p - bias),        # best possible constant correction
            "Spearman": spearmanr(t, p).correlation}                      # ranking quality (offset-free)

rows, preds = [], {}
for target in ION_ORDER:
    test = ions == target
    for setting in ["known hosts", "new hosts (strict)"]:
        train = ~test if setting == "known hosts" else (~test) & ~np.isin(groups, list(set(groups[test])))
        if train.sum() < 100:
            continue
        p_direct = new_model().fit(X[train], y[train]).predict(X[test])
        p_shift = new_model().fit(X[train], (y + shift)[train]).predict(X[test]) - E0[target]
        for method, p in [("Direct", p_direct), ("E°-shifted", p_shift)]:
            rows.append({"target ion": target, "setting": setting, "method": method,
                         "n_train": int(train.sum()), "n_test": int(test.sum()), **evaluate(y[test], p)})
            preds[(target, setting, method)] = p
    print(f"{target} done")
zs = pd.DataFrame(rows)
mae = zs.pivot_table(index="target ion", columns=["setting", "method"], values="MAE").reindex(ION_ORDER)
print("Zero-shot MAE (V):"); mae.round(3)

In [ ]:
kh = zs[zs["setting"] == "known hosts"].set_index(["target ion", "method"])
summary = pd.DataFrame({
    "MAE direct": kh.xs("Direct", level="method")["MAE"],
    "MAE E°-shifted": kh.xs("E°-shifted", level="method")["MAE"],
    "bias direct": kh.xs("Direct", level="method")["bias"],
    "bias E°-shifted": kh.xs("E°-shifted", level="method")["bias"],
    "oracle offset MAE": kh.xs("Direct", level="method")["MAE_oracle_offset"],
    "Spearman (ranking)": kh.xs("Direct", level="method")["Spearman"],
}).reindex(ION_ORDER)
summary["share of excess error removed"] = ((summary["MAE direct"] - summary["MAE E°-shifted"]) /
                                            (summary["MAE direct"] - summary["oracle offset MAE"])).clip(-2, 2)
print("Known-hosts setting. 'share removed' = how much of the removable offset error the E° shift fixes (1 = all).")
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
summary[["MAE direct", "MAE E°-shifted", "oracle offset MAE"]].plot.bar(
    ax=ax, color=["#FC8D62", "#66C2A5", "#B3B3B3"], width=0.8)
ax.set(xlabel="Ion left out of training", ylabel="Zero-shot MAE (V)",
       title="Zero-shot transfer: direct vs. physics-informed E° shift (known hosts)")
ax.legend(["Direct (Week 4)", "E°-shifted (this notebook)", "Oracle constant offset (ceiling)"])
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig("fig_e0_shift_zero_shot.png", dpi=200); plt.show()

fig, axs = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for col, t in enumerate(["Zn", "Ca", "Li"]):
    m = ions == t
    for row, method in enumerate(["Direct", "E°-shifted"]):
        a = axs[row, col]; p = preds[(t, "known hosts", method)]
        a.scatter(y[m], p, s=10, alpha=0.6, color=PALETTE[t])
        a.plot([-1, 7], [-1, 7], "k--", lw=1)
        r = zs[(zs["target ion"] == t) & (zs["setting"] == "known hosts") & (zs["method"] == method)].iloc[0]
        a.set(xlim=(-0.5, 6.5), ylim=(-0.5, 6.5), title=f"{t}, {method}\nMAE {r.MAE:.2f} V, bias {r.bias:+.2f} V")
        if row == 1: a.set_xlabel("DFT average voltage (V)")
        if col == 0: a.set_ylabel("Predicted voltage (V)")
plt.tight_layout(); plt.savefig("fig_e0_shift_parity.png", dpi=200); plt.show()

## 3. Does the shift change normal (in-distribution) accuracy?
A good correction should not hurt the regular model. Same 10 framework-based splits as Week 4.

In [ ]:
rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    p1 = new_model().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    p2 = new_model().fit(X.iloc[tr], (y + shift)[tr]).predict(X.iloc[te]) - shift[te]
    rows.append({"seed": seed, "Direct": mean_absolute_error(y[te], p1), "E°-shifted": mean_absolute_error(y[te], p2)})
ind = pd.DataFrame(rows)
print(ind[["Direct", "E°-shifted"]].agg(["mean", "std"]).round(3))
d = ind["E°-shifted"] - ind["Direct"]
print(f"\nChange: {d.mean():+.3f} V (± {d.std():.3f}); E°-shifted better in {int((d < 0).sum())}/10 splits")

## 4. Few-shot with the shift (Mg and Zn)
Does the shift also help when a few examples of the new ion are available?

In [ ]:
fs_rows = []
for target in ["Mg", "Zn"]:
    tgt = np.where(ions == target)[0]
    fw = np.array(sorted(set(groups[tgt])))
    for frac in [0.0, 0.05, 0.10]:
        for rep_i in range(3 if frac > 0 else 1):
            rng = np.random.default_rng(rep_i)
            chosen = set(rng.choice(fw, size=int(round(frac * len(fw))), replace=False)) if frac > 0 else set()
            add = np.array([i for i in tgt if groups[i] in chosen], dtype=int)
            held = np.array([i for i in tgt if groups[i] not in chosen], dtype=int)
            train = np.concatenate([np.where(ions != target)[0], add])
            p1 = new_model().fit(X.iloc[train], y[train]).predict(X.iloc[held])
            p2 = new_model().fit(X.iloc[train], (y + shift)[train]).predict(X.iloc[held]) - E0[target]
            fs_rows += [{"target ion": target, "fraction": frac, "method": "Direct", "MAE": mean_absolute_error(y[held], p1)},
                        {"target ion": target, "fraction": frac, "method": "E°-shifted", "MAE": mean_absolute_error(y[held], p2)}]
    print(f"{target} done")
fs = pd.DataFrame(fs_rows)
fs.pivot_table(index=["target ion", "fraction"], columns="method", values="MAE").round(3)

## Save

In [ ]:
import shutil
zs.to_csv("week4b_zero_shot_e0.csv", index=False); summary.to_csv("week4b_summary.csv")
ind.to_csv("week4b_in_distribution.csv", index=False); fs.to_csv("week4b_few_shot.csv", index=False)
for f in ["week4b_zero_shot_e0.csv", "week4b_summary.csv", "week4b_in_distribution.csv", "week4b_few_shot.csv",
          "fig_e0_shift_zero_shot.png", "fig_e0_shift_parity.png"]:
    shutil.copy(f, DATA_DIR)
print("Saved to", DATA_DIR)

## What to send me
1. The **spread before/after shift** lines (section 1)
2. The **summary table** (direct vs E°-shifted, bias, oracle, share removed)
3. The **in-distribution** change line (section 3)
4. The **few-shot** table, and the plots if you like

**How to read it:** if the E° shift brings Zn, Ca and Li close to the *oracle offset* MAE, the simple physics explains the transfer failure — a publishable finding. If it helps only partly, the remaining offset comes from effects $E^\circ$ does not capture (aqueous vs. solid-state references, ion–host interactions), which is also worth reporting.